# NDJF Cluster Validation and Significance

This notebook validates the exploratory `k = 2, 3, 4` clustering results from `Notebook 08`.

Primary goals:
- test whether the observed cluster silhouettes are stronger than shuffled-null feature structure
- test whether the cluster solutions are stable under repeated event resampling
- test whether the clusters differ on external variables that were not used directly in the clustering
- directly evaluate whether the `k = 3` split between cluster 2 and cluster 3 is statistically meaningful on outside variables
- summarize which `k` is best as a broad split, a working subtype framework, or an exploratory finer partition


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.satellite import default_modis_layers_for_date
from jpcz_catalog.subtypes import (
    benjamini_hochberg_adjust,
    compute_cluster_kruskal_wallis_table,
    compute_pairwise_mannwhitney_table,
    compute_permutation_silhouette_null,
    compute_resampled_cluster_stability,
    feature_definitions_dataframe,
    standardize_feature_table,
)

FEATURE_TABLE_PATH = Path("outputs/verification/jpcz_catalog_ndjf_objective_subtype_features.csv")
RUN_EXPORT_DIR_08 = Path("outputs/verification/objective_subtype_runs")
VALIDATION_EXPORT_DIR = Path("outputs/verification/objective_subtype_validation")
VALIDATION_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_subtype_validation_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
QUICKLOOK_DIR = Path("outputs/verification/ndjf_event_quicklooks_merged_12h")
SATELLITE_DIR = Path("outputs/verification/ndjf_event_satellite_panels_merged_12h")
OLR_DIR = Path("outputs/verification/ndjf_event_olr_panels_merged_12h")

CLUSTER_FEATURE_COLUMNS = [
    "coastal_to_jpcz_mean_convergence_ratio",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
    "front_box_max_temp_gradient_850_tminus12_to_tplus12",
    "sea_of_japan_mean_vorticity_peak_925",
]
VALIDATION_K_VALUES = [2, 3, 4]
PRIMARY_VALIDATION_K = 3
CLUSTER_METHOD = "ward"
N_PERMUTATIONS = 200
N_RESAMPLES = 200
SAMPLE_FRACTION = 0.8
RANDOM_SEED = 42
REPRESENTATIVE_EVENTS_PER_CLUSTER = 5
DISPLAY_REPRESENTATIVE_EVENTS_PER_CLUSTER = 1

EXTERNAL_VALIDATION_COLUMNS = [
    "pacific_to_jpcz_mean_convergence_ratio",
    "sea_of_japan_min_z850_anomaly_tminus12_to_tplus12",
    "pacific_box_max_temp_gradient_850_tminus12_to_tplus12",
    "jpcz_polygon_max_convergence_peak_925",
    "duration_hours",
    "event_peak_D_1e5_s-1",
]

PRIMARY_CLUSTER_LABELS = {
    1: "Cluster 1: least synoptic / weaker-background",
    2: "Cluster 2: moderately synoptic / circulation-modified",
    3: "Cluster 3: strongly synoptic / frontal / coastal-enhanced",
}

PAIRWISE_CLUSTER_PAIRS_BY_K = {
    3: [(2, 3)],
}

FEATURE_DICTIONARY = feature_definitions_dataframe()
FEATURE_UNITS = FEATURE_DICTIONARY.set_index("column_name")["units"].to_dict()


def maybe_copy_to_drive(path: Path):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        print("Copied to Drive:", drive_path)


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True


def axis_label(column_name: str) -> str:
    units = FEATURE_UNITS.get(column_name)
    if units is None or units == "unitless":
        return column_name
    return f"{column_name}\n[{units}]"


def quicklook_name_for_row(row_index: int, row: pd.Series) -> str:
    return f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{row_index:03d}.png"


def satellite_name_for_row(row_index: int, row: pd.Series) -> str | None:
    satellite_layers = default_modis_layers_for_date(pd.Timestamp(row['event_peak']).normalize())
    if not satellite_layers:
        return None
    satellite_layer = satellite_layers[0]
    layer_slug = (
        satellite_layer.replace("MODIS_", "")
        .replace("_CorrectedReflectance_TrueColor", "")
        .lower()
    )
    return f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{row_index:03d}_{layer_slug}.jpg"


def ensure_local_copy(local_path: Path, drive_subdir_name: str) -> bool:
    if local_path.exists():
        return True
    drive_path = Path(DRIVE_OUTPUT_DIR) / drive_subdir_name / local_path.name
    if not drive_path.exists():
        return False
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, local_path)
    return True


In [ ]:
paths_to_restore = [
    FEATURE_TABLE_PATH,
    RUN_EXPORT_DIR_08 / "cluster_comparison_summary.csv",
    RUN_EXPORT_DIR_08 / "cluster_medians_comparison_long.csv",
    RUN_EXPORT_DIR_08 / "cluster_counts_comparison.csv",
]
for run_k in VALIDATION_K_VALUES:
    paths_to_restore.extend(
        [
            RUN_EXPORT_DIR_08 / f"clustered_events_k{run_k}.csv",
            RUN_EXPORT_DIR_08 / f"cluster_medians_k{run_k}.csv",
            RUN_EXPORT_DIR_08 / f"cluster_counts_k{run_k}.csv",
            RUN_EXPORT_DIR_08 / f"cluster_quality_selected_k{run_k}.csv",
        ]
    )

for path in paths_to_restore:
    if not path.exists():
        restore_from_drive_cache(path)

feature_df = pd.read_csv(FEATURE_TABLE_PATH)
standardized_df, feature_means, feature_stds = standardize_feature_table(
    feature_df.copy(),
    columns=CLUSTER_FEATURE_COLUMNS,
)

clustered_runs = {}
cluster_columns = {}
cluster_counts_lookup = {}
cluster_quality_lookup = {}

for run_k in VALIDATION_K_VALUES:
    run_path = RUN_EXPORT_DIR_08 / f"clustered_events_k{run_k}.csv"
    if not run_path.exists():
        raise FileNotFoundError(f"Missing clustered run file: {run_path}")
    run_df = pd.read_csv(run_path)
    run_df = add_japan_local_time_columns(run_df)
    cluster_col = [c for c in run_df.columns if c.startswith("cluster_")][-1]
    expected = f"cluster_{CLUSTER_METHOD}_{run_k}"
    if cluster_col != expected:
        raise RuntimeError(f"Expected {expected} in {run_path.name}, found {cluster_col}")
    clustered_runs[run_k] = run_df
    cluster_columns[run_k] = cluster_col
    cluster_counts_lookup[run_k] = run_df[cluster_col].value_counts().sort_index().rename("n_events")

    quality_path = RUN_EXPORT_DIR_08 / f"cluster_quality_selected_k{run_k}.csv"
    if quality_path.exists():
        cluster_quality_lookup[run_k] = pd.read_csv(quality_path)
    else:
        cluster_quality_lookup[run_k] = None

loaded_summary_df = pd.DataFrame(
    {
        "k": VALIDATION_K_VALUES,
        "cluster_column": [cluster_columns[k] for k in VALIDATION_K_VALUES],
        "n_events": [int(cluster_counts_lookup[k].sum()) for k in VALIDATION_K_VALUES],
        "n_clusters_found": [int(cluster_counts_lookup[k].shape[0]) for k in VALIDATION_K_VALUES],
    }
)
print("Loaded cached clustering outputs from Notebook 08")
display(loaded_summary_df)

k2_cluster_col = cluster_columns[2]
k3_cluster_col = cluster_columns[3]
k2_vs_k3_nesting_counts = pd.crosstab(
    clustered_runs[2][k2_cluster_col],
    clustered_runs[3][k3_cluster_col],
    dropna=False,
)
k2_vs_k3_nesting_row_fraction = k2_vs_k3_nesting_counts.div(
    k2_vs_k3_nesting_counts.sum(axis=1),
    axis=0,
).round(3)
k2_vs_k3_nesting_counts_path = VALIDATION_EXPORT_DIR / "cluster_nesting_k2_vs_k3_counts.csv"
k2_vs_k3_nesting_fraction_path = VALIDATION_EXPORT_DIR / "cluster_nesting_k2_vs_k3_row_fraction.csv"
k2_vs_k3_nesting_counts.to_csv(k2_vs_k3_nesting_counts_path)
k2_vs_k3_nesting_row_fraction.to_csv(k2_vs_k3_nesting_fraction_path)
maybe_copy_to_drive(k2_vs_k3_nesting_counts_path)
maybe_copy_to_drive(k2_vs_k3_nesting_fraction_path)

print("\nk=2 vs k=3 nesting counts")
display(k2_vs_k3_nesting_counts)
print("\nk=2 vs k=3 nesting row fractions")
display(k2_vs_k3_nesting_row_fraction)
print(
    "A robust broad split should look nested here: the k=3 solution should mostly preserve the main k=2 split, "
    "then subdivide one side rather than inventing a completely different partition."
)
print("Saved nesting tables:")
print("-", k2_vs_k3_nesting_counts_path.name)
print("-", k2_vs_k3_nesting_fraction_path.name)


In [ ]:
legend_columns = CLUSTER_FEATURE_COLUMNS + EXTERNAL_VALIDATION_COLUMNS
dictionary_columns = ["units", "meaning", "formula", "calculation", "interpretation", "region", "time_window", "purpose"]
dictionary_index = FEATURE_DICTIONARY.set_index("column_name")
present_columns = [column_name for column_name in legend_columns if column_name in dictionary_index.index]
missing_columns = [column_name for column_name in legend_columns if column_name not in dictionary_index.index]

validation_feature_legend_df = (
    dictionary_index.loc[present_columns, dictionary_columns]
    .reset_index()
    .rename(columns={"column_name": "feature_column"})
)

supplemental_rows = {
    "duration_hours": {
        "units": "hours",
        "meaning": "Total event duration in hours from the merged NDJF event catalog.",
        "formula": "event_end - event_start, expressed in catalog hours",
        "calculation": "Use the merged-event duration already stored in the catalog row; this is not recomputed from the subtype feature fields.",
        "interpretation": "Larger values mean a longer-lived event. This is an external validation variable, not one of the four clustering-input features.",
        "region": "catalog-wide event property",
        "time_window": "whole merged event",
        "purpose": "Checks whether cluster groups differ in event persistence without using duration to build the clusters.",
    },
    "event_peak_D_1e5_s-1": {
        "units": "1e-5 s^-1",
        "meaning": "Peak 12 h polygon-mean divergence metric D, scaled by 1e5, from the original detector catalog.",
        "formula": "1e5 * min_t(D_t), where D is the polygon-mean 925 hPa divergence metric",
        "calculation": "Use the stored event-peak detector metric from the catalog. Because D is divergence, more negative values correspond to stronger polygon-mean convergence episodes.",
        "interpretation": "More negative values mean a stronger detector-defined JPCZ event. This is treated here as an external validation variable rather than a clustering-input feature.",
        "region": "original JPCZ polygon",
        "time_window": "event peak only",
        "purpose": "Checks whether the clusters differ in detector-defined event strength without letting D determine the subtype partition.",
    },
}

if missing_columns:
    supplemental_frames = []
    for column_name in missing_columns:
        metadata = supplemental_rows.get(
            column_name,
            {
                "units": "unknown",
                "meaning": f"Validation-only column {column_name}.",
                "formula": "stored catalog column",
                "calculation": "Loaded directly from the saved feature or cluster table.",
                "interpretation": "Interpret relative magnitudes in the context of the source catalog field.",
                "region": "varies",
                "time_window": "varies",
                "purpose": "External validation column not included in the subtype feature dictionary.",
            },
        )
        supplemental_frames.append(pd.DataFrame([{**{"feature_column": column_name}, **metadata}]))
    validation_feature_legend_df = pd.concat([validation_feature_legend_df] + supplemental_frames, ignore_index=True)

validation_feature_legend_df["feature_column"] = pd.Categorical(
    validation_feature_legend_df["feature_column"],
    categories=legend_columns,
    ordered=True,
)
validation_feature_legend_df = validation_feature_legend_df.sort_values("feature_column").reset_index(drop=True)
validation_feature_legend_df.insert(
    1,
    "plot_label",
    [axis_label(column_name) for column_name in validation_feature_legend_df["feature_column"]],
)
validation_feature_legend_path = VALIDATION_EXPORT_DIR / "validation_feature_legend.csv"
validation_feature_legend_df.to_csv(validation_feature_legend_path, index=False)
maybe_copy_to_drive(validation_feature_legend_path)

print("Validation feature legend")
display(validation_feature_legend_df)


In [ ]:
permutation_rows = []
fig, axes = plt.subplots(1, len(VALIDATION_K_VALUES), figsize=(5 * len(VALIDATION_K_VALUES), 4.5), sharey=True)
if len(VALIDATION_K_VALUES) == 1:
    axes = [axes]

for ax, run_k in zip(axes, VALIDATION_K_VALUES):
    permutation_df = compute_permutation_silhouette_null(
        standardized_df,
        n_clusters=run_k,
        method=CLUSTER_METHOD,
        n_permutations=N_PERMUTATIONS,
        random_seed=RANDOM_SEED,
    )
    permutation_path = VALIDATION_EXPORT_DIR / f"silhouette_permutation_null_k{run_k}.csv"
    permutation_df.to_csv(permutation_path, index=False)
    maybe_copy_to_drive(permutation_path)

    observed_score = float(permutation_df.loc[permutation_df["kind"] == "observed", "silhouette_score"].iloc[0])
    null_scores = permutation_df.loc[permutation_df["kind"] == "permuted", "silhouette_score"].to_numpy(dtype=float)
    null_mean = float(np.mean(null_scores))
    null_std = float(np.std(null_scores, ddof=1)) if len(null_scores) > 1 else float("nan")
    empirical_p_value = float(((null_scores >= observed_score).sum() + 1) / (len(null_scores) + 1))
    z_effect = float((observed_score - null_mean) / null_std) if np.isfinite(null_std) and null_std > 0 else float("nan")

    permutation_rows.append(
        {
            "k": run_k,
            "observed_silhouette": observed_score,
            "null_mean_silhouette": null_mean,
            "null_std_silhouette": null_std,
            "silhouette_minus_null_mean": observed_score - null_mean,
            "silhouette_z_effect": z_effect,
            "empirical_p_value": empirical_p_value,
            "n_permutations": N_PERMUTATIONS,
        }
    )

    ax.hist(null_scores, bins=18, color="lightgray", edgecolor="black")
    ax.axvline(observed_score, color="crimson", linewidth=2.0, label=f"Observed = {observed_score:.3f}")
    ax.set_title(f"Permutation null silhouette (k={run_k})")
    ax.set_xlabel("Silhouette score")
    ax.grid(alpha=0.2)
    ax.legend(loc="best")

axes[0].set_ylabel("Permutation count")
fig.tight_layout()
permutation_plot_path = PLOT_DIR / "permutation_null_silhouette_histograms.png"
fig.savefig(permutation_plot_path, dpi=170, bbox_inches="tight")
maybe_copy_to_drive(permutation_plot_path)
plt.show()

permutation_summary_df = pd.DataFrame(permutation_rows).sort_values("k").reset_index(drop=True)
permutation_summary_path = VALIDATION_EXPORT_DIR / "silhouette_permutation_summary.csv"
permutation_summary_df.to_csv(permutation_summary_path, index=False)
maybe_copy_to_drive(permutation_summary_path)

print("Permutation silhouette summary")
display(permutation_summary_df)


In [ ]:
stability_rows = []
stability_frames = []

for run_k in VALIDATION_K_VALUES:
    stability_df = compute_resampled_cluster_stability(
        standardized_df,
        n_clusters=run_k,
        method=CLUSTER_METHOD,
        n_resamples=N_RESAMPLES,
        sample_fraction=SAMPLE_FRACTION,
        random_seed=RANDOM_SEED,
    )
    stability_df.insert(0, "k", run_k)
    stability_frames.append(stability_df)

    stability_path = VALIDATION_EXPORT_DIR / f"cluster_stability_resamples_k{run_k}.csv"
    stability_df.to_csv(stability_path, index=False)
    maybe_copy_to_drive(stability_path)

    stability_rows.append(
        {
            "k": run_k,
            "n_resamples": N_RESAMPLES,
            "sample_fraction": SAMPLE_FRACTION,
            "mean_adjusted_rand_index": float(stability_df["adjusted_rand_index"].mean()),
            "median_adjusted_rand_index": float(stability_df["adjusted_rand_index"].median()),
            "p10_adjusted_rand_index": float(stability_df["adjusted_rand_index"].quantile(0.10)),
            "p90_adjusted_rand_index": float(stability_df["adjusted_rand_index"].quantile(0.90)),
            "mean_subset_silhouette": float(stability_df["subset_silhouette_score"].mean()),
        }
    )

stability_all_df = pd.concat(stability_frames, ignore_index=True)
stability_summary_df = pd.DataFrame(stability_rows).sort_values("k").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 5))
data = [stability_all_df.loc[stability_all_df["k"] == run_k, "adjusted_rand_index"].to_numpy() for run_k in VALIDATION_K_VALUES]
ax.boxplot(data, labels=[f"k={run_k}" for run_k in VALIDATION_K_VALUES], showmeans=True)
ax.set_ylabel("Adjusted Rand index")
ax.set_title("Cluster stability under repeated subsampling")
ax.grid(alpha=0.2)
fig.tight_layout()
stability_plot_path = PLOT_DIR / "cluster_stability_ari_boxplot.png"
fig.savefig(stability_plot_path, dpi=170, bbox_inches="tight")
maybe_copy_to_drive(stability_plot_path)
plt.show()

stability_summary_path = VALIDATION_EXPORT_DIR / "cluster_stability_summary.csv"
stability_summary_df.to_csv(stability_summary_path, index=False)
maybe_copy_to_drive(stability_summary_path)

print("Resampling stability summary")
display(stability_summary_df)


In [ ]:
external_rows = []
external_significant_frames = []
external_medians_lookup = {}

for run_k in VALIDATION_K_VALUES:
    run_df = clustered_runs[run_k]
    cluster_col = cluster_columns[run_k]
    kw_df = compute_cluster_kruskal_wallis_table(
        run_df,
        cluster_column=cluster_col,
        variables=EXTERNAL_VALIDATION_COLUMNS,
    )
    kw_df.insert(0, "k", run_k)
    kw_df["bh_q_value"] = benjamini_hochberg_adjust(kw_df["p_value"])
    external_rows.append(kw_df)

    external_path = VALIDATION_EXPORT_DIR / f"external_validation_kruskal_k{run_k}.csv"
    kw_df.to_csv(external_path, index=False)
    maybe_copy_to_drive(external_path)

    medians_df = run_df.groupby(cluster_col)[EXTERNAL_VALIDATION_COLUMNS].median().round(2)
    external_medians_lookup[run_k] = medians_df
    medians_path = VALIDATION_EXPORT_DIR / f"external_validation_medians_k{run_k}.csv"
    medians_df.to_csv(medians_path)
    maybe_copy_to_drive(medians_path)

    external_significant_frames.append(kw_df.loc[kw_df["bh_q_value"] <= 0.05].copy())

external_validation_df = pd.concat(external_rows, ignore_index=True)
external_validation_path = VALIDATION_EXPORT_DIR / "external_validation_kruskal_all_k.csv"
external_validation_df.to_csv(external_validation_path, index=False)
maybe_copy_to_drive(external_validation_path)

external_significant_df = pd.concat(external_significant_frames, ignore_index=True) if external_significant_frames else pd.DataFrame()
external_significant_path = VALIDATION_EXPORT_DIR / "external_validation_kruskal_significant.csv"
external_significant_df.to_csv(external_significant_path, index=False)
maybe_copy_to_drive(external_significant_path)

print("External-variable Kruskal-Wallis summary across k=2,3,4")
display(external_validation_df)
print(f"\nExternal-variable medians for primary validation solution k={PRIMARY_VALIDATION_K}")
display(external_medians_lookup[PRIMARY_VALIDATION_K])


In [ ]:
pairwise_frames = []
for run_k, cluster_pairs in PAIRWISE_CLUSTER_PAIRS_BY_K.items():
    run_df = clustered_runs[run_k]
    cluster_col = cluster_columns[run_k]
    pairwise_df = compute_pairwise_mannwhitney_table(
        run_df,
        cluster_column=cluster_col,
        variables=EXTERNAL_VALIDATION_COLUMNS,
        cluster_pairs=cluster_pairs,
    )
    pairwise_df.insert(0, "k", run_k)
    pairwise_df["bh_q_value"] = benjamini_hochberg_adjust(pairwise_df["p_value"])
    pairwise_df["median_difference_right_minus_left"] = pairwise_df["right_median"] - pairwise_df["left_median"]
    pairwise_frames.append(pairwise_df)

    pairwise_path = VALIDATION_EXPORT_DIR / f"pairwise_cluster_checks_k{run_k}.csv"
    pairwise_df.to_csv(pairwise_path, index=False)
    maybe_copy_to_drive(pairwise_path)

pairwise_summary_df = pd.concat(pairwise_frames, ignore_index=True) if pairwise_frames else pd.DataFrame()
pairwise_summary_path = VALIDATION_EXPORT_DIR / "pairwise_cluster_checks_all.csv"
pairwise_summary_df.to_csv(pairwise_summary_path, index=False)
maybe_copy_to_drive(pairwise_summary_path)

print("Pairwise external-variable checks for selected cluster pairs")
display(pairwise_summary_df)


In [ ]:
primary_run_df = clustered_runs[PRIMARY_VALIDATION_K].copy()
primary_cluster_col = cluster_columns[PRIMARY_VALIDATION_K]
standardized_primary_df, _, _ = standardize_feature_table(
    primary_run_df.copy(),
    columns=CLUSTER_FEATURE_COLUMNS,
)
valid_mask = standardized_primary_df.notna().all(axis=1) & primary_run_df[primary_cluster_col].notna()
standardized_primary_df = standardized_primary_df.loc[valid_mask].copy()
primary_run_df = primary_run_df.loc[valid_mask].copy()
primary_cluster_labels = primary_run_df[primary_cluster_col].astype(int)
centroids = standardized_primary_df.groupby(primary_cluster_labels).mean()

representative_records = []
for cluster_id in sorted(centroids.index):
    member_index = primary_cluster_labels.index[primary_cluster_labels == cluster_id]
    member_points = standardized_primary_df.loc[member_index]
    centroid = centroids.loc[cluster_id]
    distances = np.sqrt(((member_points - centroid) ** 2).sum(axis=1))
    top_members = distances.sort_values().head(REPRESENTATIVE_EVENTS_PER_CLUSTER)

    for rank_within_cluster, (row_index, centroid_distance) in enumerate(top_members.items(), start=1):
        row = primary_run_df.loc[row_index].copy()
        quicklook_name = quicklook_name_for_row(row_index, row)
        satellite_name = satellite_name_for_row(row_index, row)
        olr_name = quicklook_name
        representative_records.append(
            {
                "cluster_id": int(cluster_id),
                "cluster_label": PRIMARY_CLUSTER_LABELS.get(int(cluster_id), f"Cluster {int(cluster_id)}"),
                "representative_rank": int(rank_within_cluster),
                "catalog_index": int(row_index),
                "centroid_distance": float(centroid_distance),
                "event_peak_utc": pd.Timestamp(row["event_peak"]),
                "event_peak_jst": pd.Timestamp(row["event_peak_jst"]) if pd.notna(row.get("event_peak_jst")) else pd.NaT,
                "duration_hours": float(row["duration_hours"]),
                "event_peak_D_1e5_s-1": float(row["event_peak_D_1e5_s-1"]),
                "coastal_to_jpcz_mean_convergence_ratio": float(row["coastal_to_jpcz_mean_convergence_ratio"]),
                "sea_of_japan_mean_vorticity_peak_925": float(row["sea_of_japan_mean_vorticity_peak_925"]),
                "hokkaido_min_z850_anomaly_tminus12_to_tplus12": float(row["hokkaido_min_z850_anomaly_tminus12_to_tplus12"]),
                "front_box_max_temp_gradient_850_tminus12_to_tplus12": float(row["front_box_max_temp_gradient_850_tminus12_to_tplus12"]),
                "pattern_type_manual": row.get("pattern_type_manual", ""),
                "verification_notes": row.get("verification_notes", ""),
                "quicklook_name": quicklook_name,
                "olr_name": olr_name,
                "satellite_name": satellite_name,
            }
        )

representative_k3_df = pd.DataFrame(representative_records).sort_values(["cluster_id", "representative_rank"]).reset_index(drop=True)
representative_k3_path = VALIDATION_EXPORT_DIR / "k3_representative_events_validation.csv"
representative_k3_df.to_csv(representative_k3_path, index=False)
maybe_copy_to_drive(representative_k3_path)

print("Representative k=3 events nearest each cluster centroid")
display(representative_k3_df)

def display_validation_representative_panels(representative_df: pd.DataFrame, *, per_cluster: int = 1):
    rows_to_show = representative_df.groupby("cluster_id", group_keys=False).head(per_cluster)
    for _, rep in rows_to_show.iterrows():
        print(rep["cluster_label"])
        print(
            f"catalog idx {int(rep['catalog_index'])} | UTC {pd.Timestamp(rep['event_peak_utc']):%Y-%m-%d %H:%M} | "
            f"JST {pd.Timestamp(rep['event_peak_jst']):%Y-%m-%d %H:%M} | centroid distance {rep['centroid_distance']:.3f}"
        )

        panel_specs = [
            ("Convergence quicklook", QUICKLOOK_DIR / rep["quicklook_name"], QUICKLOOK_DIR.name),
            ("OLR panel", OLR_DIR / rep["olr_name"], OLR_DIR.name),
        ]
        if pd.notna(rep["satellite_name"]) and rep["satellite_name"]:
            panel_specs.append(("Satellite panel", SATELLITE_DIR / rep["satellite_name"], SATELLITE_DIR.name))

        fig, axes = plt.subplots(1, len(panel_specs), figsize=(6 * len(panel_specs), 6))
        if len(panel_specs) == 1:
            axes = [axes]

        for ax, (panel_label, local_path, drive_subdir_name) in zip(axes, panel_specs):
            has_panel = ensure_local_copy(local_path, drive_subdir_name)
            if not has_panel:
                ax.axis("off")
                ax.set_title(f"{panel_label}\nmissing")
                continue
            image = mpimg.imread(local_path)
            ax.imshow(image)
            ax.axis("off")
            ax.set_title(panel_label)

        fig.suptitle(rep["cluster_label"], y=0.98)
        fig.tight_layout()
        panel_path = PLOT_DIR / f"k3_representative_panels_cluster{int(rep['cluster_id'])}.png"
        fig.savefig(panel_path, dpi=170, bbox_inches="tight")
        maybe_copy_to_drive(panel_path)
        plt.show()

display_validation_representative_panels(
    representative_k3_df,
    per_cluster=DISPLAY_REPRESENTATIVE_EVENTS_PER_CLUSTER,
)


In [ ]:
comparison_summary_path = RUN_EXPORT_DIR_08 / "cluster_comparison_summary.csv"
if comparison_summary_path.exists():
    cluster_comparison_df = pd.read_csv(comparison_summary_path)
else:
    cluster_comparison_df = pd.DataFrame({"k": VALIDATION_K_VALUES})

final_summary_df = cluster_comparison_df.merge(permutation_summary_df, on="k", how="left")
final_summary_df = final_summary_df.merge(stability_summary_df, on="k", how="left")
final_summary_df["cluster_counts"] = final_summary_df["k"].map(
    lambda run_k: ", ".join(
        f"{int(cluster_id)}:{int(n_events)}" for cluster_id, n_events in cluster_counts_lookup[run_k].items()
    )
)
final_summary_df["primary_external_significant_count"] = final_summary_df["k"].map(
    lambda run_k: int((external_validation_df["k"].eq(run_k) & (external_validation_df["bh_q_value"] <= 0.05)).sum())
)
final_summary_df["recommended_role"] = ""
final_summary_df.loc[final_summary_df["k"].eq(2), "recommended_role"] = "Best broad statistical split"
final_summary_df.loc[final_summary_df["k"].eq(PRIMARY_VALIDATION_K), "recommended_role"] = "Best working subtype framework"
final_summary_df.loc[final_summary_df["k"].eq(4), "recommended_role"] = "Exploratory finer substructure"

final_summary_path = VALIDATION_EXPORT_DIR / "cluster_validation_summary.csv"
final_summary_df.to_csv(final_summary_path, index=False)
maybe_copy_to_drive(final_summary_path)

print("Final validation summary")
display(final_summary_df)
print("\nHow to read this summary:")
print("- Lower empirical permutation p-values mean the observed silhouette is less likely under shuffled-null feature structure.")
print("- Higher adjusted Rand index means the cluster solution is more stable under repeated subsampling.")
print("- External significant-counts show how many outside variables differ across clusters after BH adjustment.")
print("- k=2 is the strongest broad split if you want the simplest robust regime contrast.")
print("- k=3 is the working subtype compromise if you want one weaker and two stronger synoptic-influence groupings.")
print("- k=4 is the finer exploratory partition and should be interpreted more cautiously.")
